# "THE PRICE IS RIGHT" Capstone Project

Build a model that predicts how much something costs from a description, based on a scrape of Amazon data

## Order of play


DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 1: Data Curation

Today we'll scrub our dataset and curate our data

The dataset is here:  
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

And the folder with all the product datasets is here:  
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/raw/meta_categories

<table style="margin: 0; text-align: left;">
    <tr>        
        <td>
            <h2 style="color:#144;">Business value of Data Curation</h2>
            <span style="color:#144;">Data Curation can be considered the less glamorous work of a Data Scientist. I say that's nonsense!
            This is where the science happens - what could be more glamorous than that?! R&D with your
            dataset can often have a greater impact on performance than the fashionable 'hyper-parameter optimization' that we do later.
            So: prepare for Quality Time with Data Quality.</span>
        </td>
    </tr>
</table>

In [5]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import random
# from pricer.items import Item
# from pricer.parser import parse
load_dotenv(override=True)

True

In [6]:
# Log in to HuggingFace

hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Load our dataset

In the next cell, we load in the dataset from huggingface.

If this gives you an error like "trust_remote_code is no longer supported", then please run this command in a new cell: `!uv add --upgrade datasets==3.6.0` and then restart the Kernel, and try again.

In [8]:
dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Appliances", split="full", trust_remote_code=True)

raw/meta_categories/meta_Appliances.json(…):   0%|          | 0.00/285M [00:00<?, ?B/s]

Generating full split: 0 examples [00:00, ? examples/s]

In [9]:
print(f"Number of Appliance: {len(dataset):,}")

Number of Appliance: 94,327


In [12]:
# Investigate a particular datapoint

dataset[10]

{'main_category': 'Tools & Home Improvement',
 'title': 'WP67003405 67003405 Door Pivot Block - Compatible Kenmore KitchenAid Maytag Whirlpool Refrigerator - Replaces AP6010352 8208254 PS11743531 - Quick DIY Repair Solution',
 'average_rating': 4.1,
 'rating_number': 4,
 'features': ['WP67003405 Pivot Block For Vernicle Mullion Strip On Door - A high-quality exact equivalent for part numbers AP6010352, 67003405, 1025322, 12698403, 67003194, 8208254, and PS11743531.',
  'Compatibility with major brands - WP67003405 Door Guide is compatible with Whirlpool, Amana, Dacor, Gaggenau, Hardwick, Jenn-Air, Kenmore, KitchenAid, and Maytag.',
  "Quick DIY repair - WP67003405 Refrigerator Door Guide Pivot Block Replacement will help if your appliance door doesn't open or close. Wear work gloves to protect your hands during the repair process.",
  'Attentive support - If you are uncertain about whether the block fits your refrigerator, we will help. We generally put forth a valiant effort to guaran

In [14]:
# What's the most expensive item?

max_price = 0
max_item = None

for datapoint in tqdm(dataset):
    try:
        price = float(datapoint["price"])
        if price > max_price:
            max_price = price
            max_item = datapoint
    except ValueError:
        pass

print(f"Most expensive item: {max_item['title']} at ${max_price:.2f}")

  0%|          | 0/94327 [00:00<?, ?it/s]

Most expensive item: TurboChef BULLET Rapid Cook Electric Microwave Convection Oven at $21095.62


This is the closest I can find - looks like it's going at a bargain price!!

https://www.amazon.com/TurboChef-Electric-Countertop-Microwave-Convection/dp/B01D05U9NO/

In [ ]:
# Load into Item objects if they have a price range $1-$1000 and enough details
